# Product Review Engine

Turning raw Amazon product reviews into structured, actionable insights with an LLM (Groq API).

**Pipeline:** load data → clean → sample → extract `sentiment`, `category`, `key_insight` and `actionable_recommendation` per review → compare models.

> Dataset: [Amazon Product Reviews](https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews) (Kaggle). Put `Reviews.csv` in `../data/` (see README).


In [ ]:
import json
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

## 1. Load data

In [1]:
# Load the reviews CSV into a dataframe
df = pd.read_csv('../data/Reviews.csv')

# Output data preview
print("Dataset Preview:")
print(df.head())
df.shape

Dataset Preview:
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality

(568454, 10)

## 2. Data cleaning

In [2]:
# Are there any missing values in the dataset?
missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts[missing_counts > 0])

Missing values per column:
ProfileName    26
Summary        27
dtype: int64


In [3]:
# What do we want to do with the missing data?
# Dropping rows with no summary, as they won't affect the outcome
reviews_df = df.dropna(subset=['Summary']).copy()
# ProfileNames aren't needed for the analysis, so use a placeholder value
reviews_df['ProfileName'] = reviews_df['ProfileName'].fillna('Anonymous_User')
# Check that all missing values are gone
missing_counts = reviews_df.isna().sum()
print("Missing values per column:")
print(missing_counts[missing_counts > 0])

Missing values per column:
Series([], dtype: int64)


In [4]:
# Remove duplicates (spam reviews and reviews copied across multiple listings)
exact_duplicates = reviews_df.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")
spam_duplicates = reviews_df.duplicated(subset=['UserId', 'Time', 'Text']).sum()
print(f"Spam/Repeated review duplicates: {spam_duplicates}")
# Drop all duplicates except the first instance
reviews_df = reviews_df.drop_duplicates(subset=['UserId', 'Time', 'Text'], keep='first')

Exact duplicate rows: 0
Spam/Repeated review duplicates: 174538


In [ ]:
# Remove very short reviews that give the LLM no context (e.g. "ok", "bad")
reviews_df = reviews_df[reviews_df['Text'].str.len() > 20]

In [ ]:
# Fix a known bug for this dataset where HelpfulnessNumerator is sometimes greater than HelpfulnessDenominator
reviews_df = reviews_df[reviews_df['HelpfulnessNumerator'] <= reviews_df['HelpfulnessDenominator']]

In [ ]:
# Clean HTML tags for legibility
reviews_df['Text'] = reviews_df['Text'].str.replace(r'<br\s*/?>', ' ', regex=True)
reviews_df['Summary'] = reviews_df['Summary'].str.replace(r'<br\s*/?>', ' ', regex=True)

## 3. Sample for rapid testing

In [5]:
# Take a random sample of 50 high-quality reviews for rapid testing
sample_df = reviews_df.sample(n=50, random_state=42).reset_index(drop=True)

print(f"Sample dataset ready! Total rows: {len(sample_df)}")
display(sample_df[['Summary', 'Text', 'Score']].head())

Sample dataset ready! Total rows: 50


,Summary,Text,Score
0,Best GF fruit bar I've had yet... but it's onl...,"I'm waiting on test results, but have been giv...",4
1,Strong on the Ginger,You will really need to love the flavor of gin...,4
2,umm ummm gooooood,"great stuff, put a little in you candied sweet...",5
3,tasteless,consistency is ok but product is absolutely ta...,1
4,Nice coat less shedding,We have a 9-year old lab mix and a 1 year old ...,5


## 4. LLM extraction

The API key is read from the `GROQ_API_KEY` environment variable (`.env` file), never hardcoded.

In [6]:
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

print("Client initialized successfully!")

Client initialized successfully!


In [ ]:
def extract_review_insights(summary, text):
    prompt = f"""
    You are an FMCG Consumer Insights Specialist. 
    Analyze the customer review below and extract key commercial insights.
    
    Review Title/Summary: {summary}
    Review Body: {text}
    
    Return ONLY a valid JSON object with the following keys:
    - "sentiment": Exactly one of ["Positive", "Neutral", "Negative"]
    - "category": Choose the most relevant category from ["Packaging", "Taste & Quality", "Price & Value", "Delivery & Customer Service", "Other"]
    - "key_insight": A concise 1-sentence summary of the main customer pain point or praise.
    - "actionable_recommendation": A short 1-sentence recommendation for the brand or product management team.
    """
    
    try:
        response = client.chat.completions.create(
            model="groq/compound", 
            messages=[
                {"role": "system", "content": "You are a helpful business analytics AI that outputs only raw JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.2
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API Error: {e}")
        return {
            "sentiment": "Error",
            "category": "Error",
            "key_insight": "Error",
            "actionable_recommendation": "Error"
        }

In [ ]:
# Test the extraction on one sample review
test_summary = sample_df.iloc[0]['Summary']
test_text = sample_df.iloc[0]['Text']

print(f"Summary: {test_summary}\n")
test_result = extract_review_insights(test_summary, test_text)
print(json.dumps(test_result, indent=2))

## 5. Model comparison

Which Groq-hosted model gives the best speed and output quality for this task?

In [7]:
print([model.id for model in client.models.list().data])

['openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-86m', 'whisper-large-v3', 'qwen/qwen3.6-27b', 'allam-2-7b', 'meta-llama/llama-prompt-guard-2-22m', 'canopylabs/orpheus-arabic-saudi', 'groq/compound', 'whisper-large-v3-turbo', 'openai/gpt-oss-safeguard-20b', 'canopylabs/orpheus-v1-english', 'groq/compound-mini']


In [8]:
# Active models to compare
models_to_test = [
    "groq/compound",
    "qwen/qwen3.6-27b",
    "openai/gpt-oss-20b"
]

def benchmark_extract(model_name, summary, text):
    prompt = f"""
Analyze the following review and return ONLY a valid JSON object matching this structure:
{{
  "sentiment": "Positive" | "Negative" | "Neutral",
  "key_insight": "short summary of the key takeaway"
}}

Summary: {summary}
Review Text: {text}
"""
    start_time = time.time()
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        duration = round(time.time() - start_time, 2)
        raw_output = response.choices[0].message.content.strip()
        
        # Verify JSON validity
        try:
            parsed = json.loads(raw_output)
            output = json.dumps(parsed, indent=2, ensure_ascii=False)
        except json.JSONDecodeError:
            output = raw_output  # Fallback to raw response if model includes extra text
            
        return {"status": "success", "duration": duration, "output": output}
    except Exception as e:
        return {"status": "error", "duration": 0, "output": f"API Error: {str(e)}"}

# Select a random sample of 3 reviews from DataFrame
sample_reviews = sample_df.sample(3, random_state=42)

# Benchmark loop across all selected models and reviews
for i, (idx, row) in enumerate(sample_reviews.iterrows(), start=1):
    summary = row.get('Summary', '')
    text = row.get('Text', '')
    
    print("\n" + "=" * 50)
    print(f"SAMPLE REVIEW #{i} (Row Index {idx})")
    print(f"Summary: {summary}")
    print(f"Text: {text}")
    print("=" * 50)
    
    for model in models_to_test:
        res = benchmark_extract(model, summary, text)
        print(f"\n Model: {model}")
        print(f"⏱  Execution Time: {res['duration']}s")
        print(f" Output:\n{res['output']}")
        print("-" * 50)


SAMPLE REVIEW #1 (Row Index 13)
Summary: Perfectly light & sweet
Text: I've had various brands of agave nectar and this is my favorite.  I found it at Wal-Mart in a smaller bottle and was elated to see it on Amazon in 46 ounces.  I went from plain sugar, to Splenda, to honey, back to sugar and finally landed at agave nectar.  It is truly wonderful and light.  Great in tea and coffee.

 Model: groq/compound
⏱  Execution Time: 1.49s
 Output:
{
  "sentiment": "Positive",
  "key_insight": "Reviewer loves this agave nectar for its light, sweet taste and versatility in tea and coffee."
}
--------------------------------------------------

 Model: qwen/qwen3.6-27b
⏱  Execution Time: 3.05s
 Output:
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Input Text:** 
     - Summary: "Perfectly light & sweet"
     - Review Text: "I've had various brands of agave nectar and this is my favorite. I found it at Wal-Mart in a smaller bottle and was elated to see it on Amazon in 46 o

### Observations (3 sample reviews, informal)

- **openai/gpt-oss-20b** was the fastest (~0.3-0.4 s per review) and returned clean JSON every time.
- **groq/compound** returned valid JSON but latency varied a lot (~1.5-6.5 s).
- **qwen/qwen3.6-27b** printed its reasoning before the JSON, so strict `json.loads` fails. The package's parser (`src/product_review_engine/insights.py`) strips `<think>` blocks and extracts the JSON object to handle this.

Three reviews is a small sample, so treat this as a first impression, not a rigorous benchmark.
